# quak × boro

`quak.DataTable` is a [boro](https://github.com/manzt/boro) client: every column header is an
independently brushable view over one shared mosaic coordinator, and all of its
selection state lives in plain widget traits — so Python and the UI drive the
exact same machinery.

In [ ]:
import polars as pl
from vega_datasets import data

import quak

cars = pl.DataFrame(data.cars())
cars.head(3)

In [ ]:
d = quak.DataTable(cars)
d

Things to try in the widget above:

- **brush** a numeric or date histogram (`Horsepower`, `Year`) — every other
  column summary and the row count crossfilter, except the one you're brushing
- **click** a category bar (`Origin`) to select it; click again to clear
- **sort** with the chevron next to a column name; **scroll** for more rows
- **Reset** (bottom bar) clears every selection
- brush something, then **re-run the cell above** — the filter heals in place
  instead of leaking or resetting

## Python drives the same state

`d.col(name)` is a handle over one column's selection. Setting it from Python
moves the brush in the UI and republishes the filter — same one-way loop the
front-end uses.

In [ ]:
d.col("Horsepower").set([100, 200])
d.col("Origin").set("USA")
d.col("Horsepower")(), d.col("Origin")()

In [ ]:
d.col("Horsepower").kind, d.col("Origin").kind, d.sort

In [ ]:
# derived from the synced selection clauses + sort — the query data() runs
d.sql

In [ ]:
# the narrow waist: each column's filter arrives as an opaque SQL clause
d.selection.value

In [ ]:
d.data().pl()  # the current view (filters + sort) as a dataframe

Handles read through a [`signals`](https://github.com/manzt/signals) signal, so
they track inside `effect`/`computed` — brush the `Horsepower` column above and
watch this print:

In [ ]:
from signals import effect

hp = d.col("Horsepower")
dispose = effect(lambda: print("Horsepower selection →", hp()))

In [ ]:
dispose()
d.reset()

## Composable: share the coordinator with other boro clients

The zero-ceremony form above created a `boro.Coordinator` and a crossfilter
`boro.Selection` internally (`d.coordinator` / `d.selection` are the escape
hatches). You can also build them yourself and share them across widgets —
here's the table next to a tiny hand-written boro client, crossfiltering each
other with zero pairwise knowledge:

In [ ]:
import boro
import duckdb

con = duckdb.connect()
con.register("cars", cars.to_arrow())

coord = boro.Coordinator.connect(con)
sel = boro.Selection.crossfilter(coord)

d2 = quak.DataTable(coord, "cars", selection=sel)
d2

In [ ]:
import traitlets


class RowCounter(boro.Client):
    _esm = """
    export default {
      async render({ model, host, signal, el }) {
        el.style.cssText = "font: 14px ui-sans-serif; padding: 6px 8px;";
        el.textContent = "…";
        const coord = await host.getWidget(model.get("coord"));
        const { msql, createClient } = coord.exports;
        const ctx = await createClient({ model, host, signal });
        const rows = ctx.query(
          (filter) =>
            msql.Query.from(model.get("table"))
              .select({ n: msql.count() })
              .where(filter),
          { filterBy: ctx.filterBy },
        );
        rows.addEventListener("value", (result) => {
          if (result.isError) {
            el.textContent = `error: ${result.error.message}`;
            return;
          }
          if (!result.isSuccess) return;
          const n = Number(result.data.toColumns().n[0]);
          el.textContent = `${n.toLocaleString()} rows match`;
        }, { signal });
      },
    };
    """
    table = traitlets.Unicode().tag(sync=True)

    def __init__(self, coord, table, **kwargs):
        super().__init__(coord=coord, table=table, **kwargs)


RowCounter(coord, "cars", selection=sel)

Python can publish clauses of its own onto the shared selection — they compose
with the table's brushes and filter every client (brush a column above and
watch both filters apply):

In [ ]:
sel.update("efficient", sql='"Miles_per_Gallon" > 30')

In [ ]:
sel.update("efficient", sql=None)  # retract (same name replaces/removes)

Housekeeping, when you need it:

- `sel.value` — every live clause, with a `source` id attributing it to the
  widget/column (or `py/…` for Python-published ones)
- `sel.prune(prefix)` / `sel.clear()` — retract clauses by source id, e.g. to
  clean up after a widget you've thrown away
- `d2.data()` works here too, through the coordinator's data source